In [2]:
import pandas as pd
import os
from utils.index import (
    cat_col_to_fill,
    fill_categorical_nulls,
    impute_lot_frontage_train,
    numeric_cols,
    outliers,
    prep_data_numeric_cols,
    prep_label_data,
    sync_masonry_data,
)

df = pd.read_csv("data/train.csv")


df = df.drop(outliers).reset_index(drop=True)

df = fill_categorical_nulls(
    df,
    cat_col_to_fill,
    fill_value="None",
)

df = df.drop(columns=["Id"])

df, imp, impute_cols = impute_lot_frontage_train(df)
df, masonry_medians = sync_masonry_data(df)

df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["GarageYrBlt"].median())

df['GarageYrBlt'] = df['GarageYrBlt'].astype(str).replace('\.0', '', regex=True)

X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

X_num_col, X_num_col_scaler, cat_cols = prep_data_numeric_cols(X, numeric_cols)

X_final  = pd.concat([X_num_col, X.drop(columns=numeric_cols)], axis=1)

y_scaled, y_info = prep_label_data(y)

Detected 14 skewed columns to fix.

FIXED [LotFrontage]: Positive Skew (1.65) -> Applied Log1p
FIXED [LotArea]: Positive Skew (12.77) -> Applied Log1p
FIXED [BsmtFinSF2]: Positive Skew (4.26) -> Applied Log1p
FIXED [BsmtUnfSF]: Positive Skew (0.92) -> Applied Log1p
FIXED [1stFlrSF]: Positive Skew (0.91) -> Applied Log1p
FIXED [2ndFlrSF]: Positive Skew (0.76) -> Applied Log1p
FIXED [GrLivArea]: Positive Skew (0.87) -> Applied Log1p
FIXED [WoodDeckSF]: Positive Skew (1.56) -> Applied Log1p
FIXED [OpenPorchSF]: Positive Skew (2.40) -> Applied Log1p
FIXED [EnclosedPorch]: Positive Skew (3.07) -> Applied Log1p
FIXED [3SsnPorch]: Positive Skew (10.25) -> Applied Log1p
FIXED [ScreenPorch]: Positive Skew (4.16) -> Applied Log1p
FIXED [PoolArea]: Positive Skew (17.46) -> Applied Log1p
FIXED [MasVnrArea]: Positive Skew (2.60) -> Applied Log1p


In [3]:
from sklearn.model_selection import train_test_split


X_train, X_val, y_train, y_val = train_test_split(X_final, y_scaled, test_size=0.2, random_state=42)

In [4]:
cat_feature_indices = [X_train.columns.get_loc(col) for col in cat_cols]

print(cat_feature_indices)

X_train.columns[65]

[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78]


'GarageYrBlt'

In [5]:
from catboost import CatBoostRegressor


cat_feature_indices = [X_train.columns.get_loc(col) for col in cat_cols]

# params = {
#     "task_type": "GPU",
#     "loss_function": "RMSE",
#     "eval_metric": "RMSE",
#     "iterations": 3000,
#     "learning_rate": 0.03,
#     "depth": 4,  # Slightly shallower to prevent overfitting
#     "l2_leaf_reg": 5,  # Regularization helps with the XGB/RF gap
#     "bootstrap_type": "Bernoulli",
#     "subsample": 0.8,  # Adds randomness to each tree
#     "random_seed": 42,
#     "early_stopping_rounds": 100,
#     "verbose": 200,  # See progress every 200 iterations
# }

params = {
    "task_type": "GPU",
    "iterations": 5000,
    "learning_rate": 0.02,  # Very slow learning
    "depth": 6,  # Simple trees
    "l2_leaf_reg": 10,  # High penalty for complex leaves
    "random_strength": 2,  # Adds noise to split selection
    "min_data_in_leaf": 15,  # Prevents leaves from capturing single outliers
    "bagging_temperature": 1,
    "early_stopping_rounds": 500,
    "verbose": 500,
}

model = CatBoostRegressor(**params)

model.fit(
    X_train,
    y_train,
    cat_features=cat_feature_indices,
    eval_set=(X_val, y_val),
    early_stopping_rounds=500,
)

0:	learn: 0.9969675	test: 0.9575876	best: 0.9575876 (0)	total: 124ms	remaining: 10m 20s
500:	learn: 0.2925191	test: 0.3325707	best: 0.3325707 (500)	total: 59.1s	remaining: 8m 50s
1000:	learn: 0.2532456	test: 0.3234294	best: 0.3233400 (995)	total: 1m 56s	remaining: 7m 43s
1500:	learn: 0.2383787	test: 0.3228387	best: 0.3225655 (1175)	total: 2m 53s	remaining: 6m 44s
bestTest = 0.3225655299
bestIteration = 1175
Shrink model to first 1176 iterations.


In [6]:

from utils.index import impute_lot_frontage_test

def prep_test_data(test_df):
    # 1. ID Handling
    id_col = test_df["Id"].copy()
    test_df = test_df.drop(columns=["Id"])

    # 2. Fill Categorical/Masonry using Training Logic
    test_df = fill_categorical_nulls(test_df, cat_col_to_fill, fill_value="None")

    # LotFrontage Regression using fitted imputer
    test_df = impute_lot_frontage_test(test_df, imp, impute_cols)

    # Masonry using training medians
    test_df, _ = sync_masonry_data(test_df, masonry_medians=masonry_medians)

    # 3. Simple Null Filling from Training Constants
    test_df["Electrical"] = test_df["Electrical"].fillna(df["Electrical"].mode()[0])
    test_df["GarageYrBlt"] = test_df["GarageYrBlt"].fillna(df["GarageYrBlt"].mode()[0])

    # Fill any remaining numeric NaNs (e.g., GarageCars, TotalBsmtSF)
    test_df[numeric_cols] = test_df[numeric_cols].fillna(
        df[numeric_cols].median()
    )

    # 4. Scaling
    X_test_num_scaled = pd.DataFrame(
        X_num_col_scaler.transform(test_df[numeric_cols]),
        columns=numeric_cols,
        index=test_df.index,
    )

    # Force test columns to match training columns exactly
    X_test_final = X_test_num_scaled.reindex(columns=X_train.columns.tolist(), fill_value=0)

    return id_col, X_test_final

test_df = pd.read_csv('data/test.csv')

id_col, X_test_final = prep_test_data(test_df)

In [7]:
from utils.index import revert_label_data

preds_scaled = model.predict(X_test_final)
final_prices = revert_label_data(preds_scaled, y_info)

submission = pd.DataFrame({
    'Id': id_col,
    'SalePrice': final_prices
})

model_name = type(model).__name__
submission.to_csv(f'{model_name}_submission.csv', index=False)